[![Roboflow Notebooks](https://media.roboflow.com/notebooks/template/bannertest2-2.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672932710194)](https://github.com/roboflow/notebooks)

# How to Tell Objects Apart with ReID

Appearance re-identification (ReID) embeds object crops so you can tell lookalikes
apart across frames or cameras. In this notebook you will load pretrained encoders
with the [`reid`](https://github.com/roboflow/re-ID) package, score Market-1501 with
`ReIDEvaluator`, then visualize SportsMOT soccer crops: gallery ranking and
same-player vs different-player decisions from cosine distance.

For metric definitions and cosine vs euclidean distance, see the
[`reid` metrics guide](https://reid.roboflow.com/latest/learn/metrics/).

## Setup

### Check GPU availability

Let's make sure that we have access to GPU. We can use `nvidia-smi` command to do
that. In case of any problems navigate to `Runtime` -> `Change runtime type` ->
`Hardware accelerator`, set it to `GPU`, and then click `Save`.

In [ ]:
!nvidia-smi

### Install dependencies

Install `reid` from PyPI. `trackers` is only used later to download SportsMOT for
the sports visualization section.

You may see dependency conflict warnings in Google Colab. This is expected for the
preinstalled Google Colab environment and does not affect functionality.

In [ ]:
!pip install -q --upgrade pip
!pip install -q reid matplotlib gdown trackers

In [ ]:
import zipfile
from pathlib import Path

import cv2
import gdown
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

from reid import ReIDEvaluator, ReIDModel
from reid.data import build_identity_crop_retrieval_split, load_identity_crops

ROOT = Path(".")
CKPT_DIR = ROOT / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()} | {device}")

def download_checkpoint(gdrive_id: str, filename: str) -> str:
    """Download an OSNet checkpoint from the torchreid model zoo (Google Drive)."""
    path = CKPT_DIR / filename
    if not path.exists():
        gdown.download(id=gdrive_id, output=str(path), quiet=False)
    return str(path)

def print_comparison(name, cos, euc, zoo_r1, zoo_map):
    """Print a cosine-vs-euclidean comparison table against model-zoo targets."""
    print(f"\n{name}: distance metric comparison")
    print(f"{'metric':<10}{'cosine':>10}{'euclidean':>12}{'model zoo':>12}")
    print("-" * 44)
    print(f"{'Rank-1':<10}{cos.rank1:>9.1f}%{euc.rank1:>11.1f}%{zoo_r1:>11.1f}%")
    print(f"{'mAP':<10}{cos.mean_average_precision:>9.1f}%{euc.mean_average_precision:>11.1f}%{zoo_map:>11.1f}%")
    print(f"{'Rank-5':<10}{cos.rank5:>9.1f}%{euc.rank5:>11.1f}%{'-':>12}")
    print(f"{'Rank-10':<10}{cos.rank10:>9.1f}%{euc.rank10:>11.1f}%{'-':>12}")
    print(f"{'mINP':<10}{cos.minp:>9.1f}%{euc.minp:>11.1f}%{'-':>12}")

## Market-1501 metrics

**Expected (OSNet x1.0, Market-trained, model zoo):** R1 ≈ 94.2 / mAP ≈ 82.6 (euclidean).

Download Market-1501, load the query and gallery splits, then evaluate a Market-trained
OSNet checkpoint. We also report cosine (L2-normalized embeddings, the package default)
so you can see the gap versus the published euclidean protocol.

In [ ]:
MARKET_ZIP = ROOT / "Market-1501.zip"
MARKET_DIR = ROOT / "Market-1501-v15.09.15"

if not MARKET_DIR.exists():
    gdown.download(id="0B8-rUzbwVRk0c054eEozWG9COHM", output=str(MARKET_ZIP), quiet=False)
    with zipfile.ZipFile(MARKET_ZIP, "r") as zf:
        zf.extractall(ROOT)
    print("Extracted to", MARKET_DIR)
else:
    print("Already downloaded:", MARKET_DIR)

In [ ]:
from reid import load_market1501

query_m, gallery_m = load_market1501(str(MARKET_DIR))
print(f"Market-1501: {len(query_m):,} query, {len(gallery_m):,} gallery")

In [ ]:
market_ckpt = download_checkpoint("1vduhq5DpN2q1g4fYEZfPI17MJeh9qyrA", "osnet_x1_0_market1501.pth")
model_market = ReIDModel.from_pretrained(market_ckpt, architecture="osnet_x1_0")
evaluator_market = ReIDEvaluator(model_market, batch_size=256)

result_market = evaluator_market.evaluate(query_m, gallery_m, distance="cosine", return_distmat=False)
result_market_euc = evaluator_market.evaluate(
    query_m,
    gallery_m,
    distance="euclidean",
    return_distmat=False,
    query_embeddings=result_market.query_embeddings,
    gallery_embeddings=result_market.gallery_embeddings,
)

print_comparison("Market-1501", result_market.metrics, result_market_euc.metrics, 94.2, 82.6)

## Sports ranking visualization (SportsMOT)

Download one SportsMOT val sequence, build identity crops, run retrieval with the
default pretrained OSNet, and plot a few query to top gallery rankings. Green titles
are correct identity matches; red are distractors.

In [ ]:
SPORTSMOT_ROOT = ROOT / "sportsmot"
SPORTSMOT_VAL = SPORTSMOT_ROOT / "val"
CROPS_DIR = ROOT / "sportsmot_reid_crops"

!trackers download sportsmot --split val --asset annotations,frames -o {ROOT}

def generate_mot_patches(mot_root, output_dir, *, split="full", sequences=None, min_visibility=0.0, min_side=0):
    """Crop MOT GT boxes to output_dir/<seq>_<id>/<seq>_<frame>.jpg.

    Slim notebook helper for SportsMOT ranking. Published reid does not ship MOT
    crop generation yet, so we keep a local patch extractor here.
    """
    if split != "full":
        raise ValueError("This notebook helper only supports split='full'.")

    mot_root = Path(mot_root)
    output_dir = Path(output_dir)
    if sequences is None:
        sequences = sorted(
            p.name for p in mot_root.iterdir()
            if p.is_dir() and (p / "gt" / "gt.txt").is_file() and (p / "img1").is_dir()
        )
    if not sequences:
        raise FileNotFoundError(f"No MOT sequences under {mot_root}")

    num_crops = 0
    identities = set()
    for seq in sequences:
        seq_dir = mot_root / seq
        rows = np.loadtxt(seq_dir / "gt" / "gt.txt", delimiter=",", ndmin=2)
        if rows.size == 0:
            continue
        # frame, id, x, y, w, h, conf, class, visibility
        keep = (rows[:, 7].astype(int) == 1) & (rows[:, 6] > 0)
        rows = rows[keep]
        frame_cache = {}
        for row in rows:
            visibility = float(row[8]) if row.shape[0] > 8 else 1.0
            if visibility < min_visibility:
                continue
            frame = int(row[0])
            if frame not in frame_cache:
                img_path = None
                for width in (6, 8):
                    candidate = seq_dir / "img1" / f"{frame:0{width}d}.jpg"
                    if candidate.exists():
                        img_path = candidate
                        break
                frame_cache[frame] = cv2.imread(str(img_path)) if img_path is not None else None
            image = frame_cache[frame]
            if image is None:
                continue
            h_img, w_img = image.shape[:2]
            x, y, w, h = (float(v) for v in row[2:6])
            x1, y1 = max(0, round(x)), max(0, round(y))
            x2, y2 = min(w_img, round(x + w)), min(h_img, round(y + h))
            if x2 <= x1 or y2 <= y1:
                continue
            if min_side > 0 and min(x2 - x1, y2 - y1) < min_side:
                continue
            identity = f"{seq}_{int(row[1])}"
            identity_dir = output_dir / identity
            identity_dir.mkdir(parents=True, exist_ok=True)
            crop_path = identity_dir / f"{seq}_{frame:06d}.jpg"
            if not cv2.imwrite(str(crop_path), image[y1:y2, x1:x2]):
                raise OSError(f"Failed to write crop to {crop_path}")
            num_crops += 1
            identities.add(identity)

    return {"sequences": list(sequences), "num_identities": len(identities), "num_crops": num_crops}

SPORTS_SEQS = sorted(p.name for p in SPORTSMOT_VAL.iterdir() if (p / "gt" / "gt.txt").is_file())[:1]
print("Using sequences:", SPORTS_SEQS)

stats = generate_mot_patches(
    SPORTSMOT_VAL,
    CROPS_DIR,
    split="full",
    sequences=SPORTS_SEQS,
)
print(stats)

### Embed crops and run retrieval

Load identity folders, build a query / gallery split, and evaluate with cosine
distance. SportsMOT numbers here are illustrative only (synthetic camids), not a
model-zoo target.

In [ ]:
# Default combineall OSNet is fine here: we only want a qualitative sports ranking demo.
model_sports = ReIDModel.from_pretrained()
evaluator_sports = ReIDEvaluator(model_sports, batch_size=128)

crops = load_identity_crops(CROPS_DIR)
query_s, gallery_s = build_identity_crop_retrieval_split(
    crops, queries_per_id=1, min_crops=4, seed=0
)
print(f"SportsMOT crops: {len(crops)} total -> {len(query_s)} query / {len(gallery_s)} gallery")

result_sports = evaluator_sports.evaluate(
    query_s, gallery_s, distance="cosine", return_distmat=True, verbose=False
)
m = result_sports.metrics
print(
    f"SportsMOT retrieval (synthetic camids): "
    f"mAP {m.mean_average_precision:.1f}%  Rank-1 {m.rank1:.1f}%  "
    f"Rank-5 {m.rank5:.1f}%  (illustrative, not a zoo target)"
)

### Plot top gallery rankings

For a few queries, show the query crop and its top-k gallery neighbors ranked by
cosine distance.

In [ ]:
TOP_K = 5
N_QUERIES = 4

def _load_rgb(path: str) -> np.ndarray:
    image = Image.open(path).convert("RGB")
    return np.asarray(image)

# Prefer queries that have at least one true match in the top-k when possible.
order = np.argsort(result_sports.distmat, axis=1)
shown = 0
fig, axes = plt.subplots(N_QUERIES, TOP_K + 1, figsize=(2.2 * (TOP_K + 1), 2.6 * N_QUERIES))
if N_QUERIES == 1:
    axes = np.expand_dims(axes, 0)

for q_idx in range(len(query_s)):
    if shown >= N_QUERIES:
        break
    ranks = order[q_idx]
    q_pid = int(query_s.pids[q_idx])
    has_hit = any(int(gallery_s.pids[g]) == q_pid for g in ranks[:TOP_K])
    if not has_hit and q_idx < len(query_s) - N_QUERIES:
        continue

    row = axes[shown]
    row[0].imshow(_load_rgb(query_s.image_paths[q_idx]))
    row[0].set_title(f"query\nid {q_pid}", fontsize=9)
    row[0].axis("off")
    for j, g_idx in enumerate(ranks[:TOP_K]):
        g_pid = int(gallery_s.pids[g_idx])
        match = g_pid == q_pid
        row[j + 1].imshow(_load_rgb(gallery_s.image_paths[g_idx]))
        row[j + 1].set_title(
            f"{'MATCH' if match else 'distractor'}\nid {g_pid}\nd={result_sports.distmat[q_idx, g_idx]:.2f}",
            fontsize=8,
            color="#1a7f37" if match else "#b42318",
        )
        row[j + 1].axis("off")
    shown += 1

for row in axes[shown:]:
    for ax in row:
        ax.axis("off")

fig.suptitle(f"SportsMOT ranking · {SPORTS_SEQS[0]} · cosine top-{TOP_K}", y=1.01)
fig.tight_layout()
plt.show()

## Same player or not?

Sample random SportsMOT crop pairs (half same identity, half different). Same-player
pairs use frames at least `MIN_FRAME_GAP` apart so the comparison is not near-duplicate
consecutive crops. Embed both crops, compute cosine distance, and decide **same player**
when `distance < THRESHOLD`. Green titles match ground truth; red do not.

`THRESHOLD` is illustrative for this demo. Tune it for your domain (tracking often
uses a lower cosine threshold such as 0.2).

In [ ]:
from collections import defaultdict

def _load_rgb(path: str) -> np.ndarray:
    return np.asarray(Image.open(path).convert("RGB"))

def _frame_index(path: str) -> int:
    # Parse frame id from crop names like seq_000123.jpg
    return int(Path(path).stem.rsplit("_", 1)[-1])

N_PAIRS = 8
THRESHOLD = 0.30  # cosine distance; same player if distance < THRESHOLD
MIN_FRAME_GAP = 60  # require temporally distant same-player crops
SEED = 1

by_pid: dict[int, list[str]] = defaultdict(list)
for path_str, pid in zip(crops.image_paths, crops.pids):
    by_pid[int(pid)].append(path_str)

def same_player_candidates(paths: list[str]) -> list[tuple[str, str, int]]:
    """Return (path_a, path_b, frame_gap) pairs with gap >= MIN_FRAME_GAP."""
    items = sorted((_frame_index(p), p) for p in paths)
    out: list[tuple[str, str, int]] = []
    for i, (f1, p1) in enumerate(items):
        for f2, p2 in items[i + 1 :]:
            gap = f2 - f1
            if gap >= MIN_FRAME_GAP:
                out.append((p1, p2, gap))
    return out

same_pids = [pid for pid, paths in by_pid.items() if same_player_candidates(paths)]
diff_pids = list(by_pid.keys())
if len(same_pids) < 1 or len(diff_pids) < 2:
    raise RuntimeError(
        f"Need identities with crops >= {MIN_FRAME_GAP} frames apart; "
        f"found {len(same_pids)} eligible same-player ids."
    )

rng = np.random.default_rng(SEED)
pairs: list[tuple[str, str, bool, int]] = []

n_same = N_PAIRS // 2
for _ in range(n_same):
    pid = int(rng.choice(same_pids))
    candidates = same_player_candidates(by_pid[pid])
    # Prefer larger temporal gaps so pose/viewpoint actually change.
    candidates.sort(key=lambda item: item[2], reverse=True)
    top = candidates[: max(1, len(candidates) // 3)]
    path_a, path_b, gap = top[int(rng.integers(0, len(top)))]
    pairs.append((path_a, path_b, True, gap))

for _ in range(N_PAIRS - n_same):
    pid_a, pid_b = (int(x) for x in rng.choice(diff_pids, size=2, replace=False))
    path_a = str(rng.choice(by_pid[pid_a]))
    path_b = str(rng.choice(by_pid[pid_b]))
    gap = abs(_frame_index(path_a) - _frame_index(path_b))
    pairs.append((path_a, path_b, False, gap))

rng.shuffle(pairs)

unique_paths = sorted({p for pair in pairs for p in pair[:2]})
embeddings = model_sports.extract_features_from_paths(unique_paths)
path_to_emb = {p: embeddings[i] for i, p in enumerate(unique_paths)}

def cosine_distance(a: np.ndarray, b: np.ndarray) -> float:
    a = a / (np.linalg.norm(a) + 1e-12)
    b = b / (np.linalg.norm(b) + 1e-12)
    return float(1.0 - np.dot(a, b))

fig, axes = plt.subplots(N_PAIRS, 2, figsize=(5.2, 2.4 * N_PAIRS))
if N_PAIRS == 1:
    axes = np.expand_dims(axes, 0)

correct = 0
for row_idx, (path_a, path_b, same_gt, gap) in enumerate(pairs):
    dist = cosine_distance(path_to_emb[path_a], path_to_emb[path_b])
    same_pred = dist < THRESHOLD
    ok = same_pred == same_gt
    correct += int(ok)
    color = "#1a7f37" if ok else "#b42318"
    decision = "SAME PLAYER" if same_pred else "DIFFERENT"
    truth = "same" if same_gt else "different"
    title = f"{decision}  d={dist:.2f}  Δf={gap}  (gt: {truth})"

    for col, crop_path in enumerate((path_a, path_b)):
        axes[row_idx, col].imshow(_load_rgb(crop_path))
        axes[row_idx, col].axis("off")
        if col == 0:
            axes[row_idx, col].set_title(title, fontsize=9, color=color, loc="left")

fig.suptitle(
    f"Same player?  threshold={THRESHOLD:.2f}  min Δf={MIN_FRAME_GAP}  "
    f"accuracy={correct}/{N_PAIRS}",
    y=1.01,
)
fig.tight_layout()
plt.show()
print(
    f"Correct decisions: {correct}/{N_PAIRS} at cosine threshold {THRESHOLD:.2f} "
    f"(same-player min frame gap {MIN_FRAME_GAP})"
)

## Results

Print the Market-1501 cosine and euclidean metrics side by side. Numbers should land
near the model-zoo targets (euclidean).

In [ ]:
print(f"{'Dataset':<14}{'encoder':<22}{'distance':<12}{'mAP':>8}{'Rank-1':>9}{'Rank-5':>9}{'Rank-10':>9}{'mINP':>8}")
print("-" * 91)

def _row(name, encoder, dist, m):
    return (
        f"{name:<14}{encoder:<22}{dist:<12}{m.mean_average_precision:>7.1f}%{m.rank1:>8.1f}%"
        f"{m.rank5:>8.1f}%{m.rank10:>8.1f}%{m.minp:>7.1f}%"
    )

print(_row("Market-1501", "OSNet", "cosine", result_market.metrics))
print(_row("", "", "euclidean", result_market_euc.metrics))
print("-" * 91)
print("Model-zoo target (OSNet, euclidean): Market-1501 R1≈94.2 mAP≈82.6")

You just scored Market-1501 and ran SportsMOT ReID visualizations. Nice work!

The `reid` package makes it easy to load pretrained encoders, evaluate gallery metrics,
and compare crops by appearance distance.

Ready to go deeper? Explore the [`reid` package](https://github.com/roboflow/re-ID),
the [metrics guide](https://reid.roboflow.com/latest/learn/metrics/), or wire appearance
into tracking with
[How to Add ReID to Trackers](https://colab.research.google.com/github/roboflow-ai/notebooks/blob/main/notebooks/how-to-add-reid-to-trackers.ipynb).

Got feedback or ideas? Open an issue on
[GitHub Issues](https://github.com/roboflow/re-ID/issues).